# profiles_pkl_to_csv

Convert the VQ-VAE **profiles pickle** into the per-user embedding-sequence CSV used by
`profiles_to_dcabp.ipynb`.


In [1]:
import pickle
import sys
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    cwd = Path.cwd()
    if cwd.name == "notebooks":
        return cwd.parent
    if cwd.name == "vq-vae_lda_pipeline":
        return cwd.parent.parent
    for p in (cwd, *cwd.parents):
        if (p / "models").is_dir() and (p / "data").is_dir():
            return p
    return cwd


def rel_path(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


PROJECT_ROOT = find_project_root()

PROFILES_PKL = PROJECT_ROOT / "data/output_vq_vae/profiles_per_sample_oncology.pkl"
OUTPUT_CSV = PROJECT_ROOT / "data/processed/lda/user_embeddings_from_pkl.csv"

MODEL_TYPE = "a0"  # a0 | a1 | a2
N = 30  # profile length key in the PKL (5, 10, 15, 20, 25, 30)

## 1 · Load the profiles PKL

In [2]:
if not PROFILES_PKL.is_file():
    raise FileNotFoundError(
        f"PKL not found: {rel_path(PROFILES_PKL)}. "
        "Run notebooks/vq-vae_lda_pipeline/daily_to_profiles.ipynb first."
    )

with open(PROFILES_PKL, "rb") as f:
    embedding_counts = pickle.load(f)

print("Loaded:", rel_path(PROFILES_PKL))
print("Top-level keys:", list(embedding_counts.keys()))


Loaded: data/output_vq_vae/profiles_per_sample_oncology.pkl
Top-level keys: ['a0', 'a1', 'a2']


## 2 · Inspect structure (model type and `n`)

In [3]:
if MODEL_TYPE not in embedding_counts:
    raise KeyError(
        f"model_type={MODEL_TYPE!r} not in PKL. Available: {list(embedding_counts.keys())}"
    )

available_n = sorted(embedding_counts[MODEL_TYPE].keys())
print(f"model_type={MODEL_TYPE!r} → available n:", available_n)

if N not in embedding_counts[MODEL_TYPE]:
    raise KeyError(f"n={N} not available. Choose one of {available_n}")

users_at_n = embedding_counts[MODEL_TYPE][N]
print(f"Patients at n={N}:", len(users_at_n))

# Example patient (same check as 4_01_LDA_dec25.ipynb)
example_user = next(iter(users_at_n))
user_embed = users_at_n[example_user]
print(f"Example user_id={example_user}, payload type={type(user_embed)}")
print(f"  sequence length: {user_embed[0][0]}")
print(f"  embedding_ids length: {len(user_embed[0][1])}")

model_type='a0' → available n: [5, 10, 15, 20, 25, 30]
Patients at n=30: 175
Example user_id=11001, payload type=<class 'list'>
  sequence length: 877
  embedding_ids length: 877


## 3 · Build DataFrame and save CSV

In [4]:
user_data = []
for user_id, payload in embedding_counts[MODEL_TYPE][N].items():
    user_embed = payload[0]
    embedding_ids = user_embed[1]
    user_data.append(
        {
            "user_id": user_id,
            "embedding_ids": embedding_ids.tolist(),
        }
    )

df = pd.DataFrame(user_data).sort_values("user_id").reset_index(drop=True)

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)

print("Saved:", rel_path(OUTPUT_CSV))
print("Rows:", len(df))
df.head()

Saved: data/processed/lda/user_embeddings_from_pkl.csv
Rows: 175


,user_id,embedding_ids
0,11001,"[67, 34, 70, 7, 76, 125, 95, 62, 131, 8, 184, ..."
1,11003,"[199, 13, 184, 225, 192, 224, 72, 84, 220, 8, ..."
2,11004,"[43, 85, 218, 210, 59, 84, 243, 125, 125, 218,..."
3,11005,"[69, 192, 50, 110, 131, 117, 243, 67, 149, 14,..."
4,11006,"[85, 208, 250, 113, 125, 208, 208, 15, 247, 51..."
